<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/T1/1_Data_collection_RecA_publication_ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Collection and Curation of *Mycobacterium tuberculosis* RecA EC50 Bioactivity Data from ChEMBL

This notebook is a publication-ready data collection workflow adapted from the original dengue virus type 2 NS3 protein notebook and combined with the RecA ChEMBL EC50 workflow.

**Target used in this notebook**

- Target: *Mycobacterium tuberculosis* RecA protein
- ChEMBL target ID: `CHEMBL1741171`
- Bioactivity endpoint: `EC50`
- Standard unit: `nM`
- Accepted relation: exact values only (`standard_relation == "="`)

**Main outputs**

The workflow generates raw, filtered, curated, labeled, binary, balanced, and summary CSV files inside:

`outputs/recA_chembl/`


## 1. Install required libraries

In [10]:
# Run this cell in Google Colab if the ChEMBL client is not installed.
# In local Jupyter, you may also run it once.
!pip -q install chembl_webresource_client pandas


## 2. Import libraries and define configuration

In [11]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Iterable

import pandas as pd
from chembl_webresource_client.new_client import new_client


# ============================================================
# Project configuration
# ============================================================

TARGET_SEARCH_QUERY = "RecA Mycobacterium tuberculosis"
TARGET_CHEMBL_ID = "CHEMBL1741171"
TARGET_PREF_NAME = "Protein RecA"
TARGET_ORGANISM = "Mycobacterium tuberculosis"

STANDARD_TYPE = "EC50"
STANDARD_UNIT = "nM"
STANDARD_RELATION = "="

ACTIVE_THRESHOLD_NM = 1000
INACTIVE_THRESHOLD_NM = 10000
RANDOM_STATE = 42
MOLECULE_BATCH_SIZE = 100

OUTPUT_DIR = Path("outputs") / "recA_chembl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Search and select the RecA target

This section follows the same logic as the original NS3 notebook: first search the ChEMBL target table, then select the correct biological target.  
For publication-quality reproducibility, this notebook does **not** rely only on the first search result. It prioritizes the validated ChEMBL target ID `CHEMBL1741171`.


In [12]:
def search_targets(query: str = TARGET_SEARCH_QUERY) -> pd.DataFrame:
    """Search ChEMBL target records using a keyword query."""
    target_client = new_client.target
    records = list(target_client.search(query))

    if not records:
        raise RuntimeError(f"No ChEMBL target records found for query: {query}")

    targets = pd.DataFrame.from_dict(records)
    targets.to_csv(OUTPUT_DIR / "recA_target_search_hits.csv", index=False)
    return targets


def select_reca_target(targets: pd.DataFrame) -> pd.Series:
    """Select the intended M. tuberculosis RecA target from ChEMBL search results."""
    required_columns = {"target_chembl_id", "pref_name", "organism"}
    missing = required_columns.difference(targets.columns)

    if missing:
        raise ValueError(f"Target search result is missing columns: {sorted(missing)}")

    selected = targets[targets["target_chembl_id"] == TARGET_CHEMBL_ID]

    if selected.empty:
        selected = targets[
            (targets["pref_name"].astype(str).str.contains("RecA", case=False, na=False))
            & (targets["organism"].astype(str) == TARGET_ORGANISM)
        ]

    if selected.empty:
        raise RuntimeError(
            "The expected RecA target was not found. "
            "Please inspect outputs/recA_chembl/recA_target_search_hits.csv."
        )

    selected_target = selected.iloc[0]
    pd.DataFrame([selected_target.to_dict()]).to_csv(
        OUTPUT_DIR / "recA_selected_target.csv",
        index=False,
    )
    return selected_target


targets = search_targets()
selected_target = select_reca_target(targets)

print("Selected target")
print("---------------")
print(f"Target ChEMBL ID : {selected_target['target_chembl_id']}")
print(f"Preferred name   : {selected_target['pref_name']}")
print(f"Organism         : {selected_target['organism']}")

targets.head()


Selected target
---------------
Target ChEMBL ID : CHEMBL1741171
Preferred name   : Protein RecA
Organism         : Mycobacterium tuberculosis


,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Mycobacterium tuberculosis,Mycobacterium tuberculosis,28.0,False,CHEMBL360,[],ORGANISM,1773
1,[],Mycobacterium tuberculosis H37Rv,Mycobacterium tuberculosis H37Rv,25.0,False,CHEMBL2111188,[],ORGANISM,83332
2,[],Mycobacterium tuberculosis variant bovis,Mycobacterium tuberculosis variant bovis,22.0,False,CHEMBL613086,[],ORGANISM,1765
3,[],Mycobacterium tuberculosis variant microti,Mycobacterium tuberculosis variant microti,22.0,False,CHEMBL612960,[],ORGANISM,1806
4,[],Mycobacterium tuberculosis,Protein RecA,22.0,False,CHEMBL1741171,"[{'accession': 'P9WHJ3', 'component_descriptio...",SINGLE PROTEIN,1773


## 4. Retrieve RecA EC50 bioactivity records

This section retrieves ChEMBL bioactivity records for the selected RecA target and keeps only exact EC50 records in nM.


In [13]:
def fetch_activities(target_chembl_id: str) -> pd.DataFrame:
    """Retrieve raw ChEMBL activity records for a target."""
    fields = [
        "activity_id",
        "assay_chembl_id",
        "assay_description",
        "canonical_smiles",
        "document_chembl_id",
        "molecule_chembl_id",
        "pchembl_value",
        "published_type",
        "published_units",
        "published_value",
        "standard_relation",
        "standard_type",
        "standard_units",
        "standard_value",
        "target_chembl_id",
    ]

    activity_client = new_client.activity
    records = list(
        activity_client
        .filter(target_chembl_id=target_chembl_id)
        .only(fields)
    )

    if not records:
        raise RuntimeError(f"No activity records found for target: {target_chembl_id}")

    raw = pd.DataFrame.from_dict(records)
    raw.to_csv(OUTPUT_DIR / "recA_activities_raw.csv", index=False)
    return raw


def filter_exact_ec50_nm(raw: pd.DataFrame) -> pd.DataFrame:
    """Keep exact EC50 records reported in nM and compute pEC50."""
    df = raw.copy()

    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
    df["pchembl_value"] = pd.to_numeric(df["pchembl_value"], errors="coerce")

    ec50 = df[
        (df["standard_type"] == STANDARD_TYPE)
        & (df["standard_units"] == STANDARD_UNIT)
        & (df["standard_relation"] == STANDARD_RELATION)
        & df["standard_value"].notna()
        & (df["standard_value"] > 0)
    ].copy()

    ec50["standard_value_nM"] = ec50["standard_value"]
    ec50["pEC50"] = ec50["standard_value_nM"].map(lambda x: 9 - math.log10(x))

    ec50.to_csv(OUTPUT_DIR / "recA_activities_ec50_exact.csv", index=False)
    return ec50


raw_activities = fetch_activities(selected_target["target_chembl_id"])
ec50_activities = filter_exact_ec50_nm(raw_activities)

print(f"Raw activity records       : {len(raw_activities)}")
print(f"Exact EC50 records in nM   : {len(ec50_activities)}")

ec50_activities.head()


Raw activity records       : 2211
Exact EC50 records in nM   : 1914


,activity_id,assay_chembl_id,assay_description,canonical_smiles,document_chembl_id,molecule_chembl_id,pchembl_value,relation,standard_relation,standard_type,standard_units,standard_value,target_chembl_id,type,units,value,standard_value_nM,pEC50
88,6720436,CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,O=C1/C(=C/c2ccccc2F)SC(=S)N1CC(=O)N1CCCCC1c1cc...,CHEMBL1201862,CHEMBL1305372,4.73,=,=,EC50,nM,18600.0,CHEMBL1741171,EC50,uM,18.6,18600.0,4.730487
89,6720437,CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,CCc1n[nH]c2c1C(c1ccsc1)C(C#N)=C(N)O2,CHEMBL1201862,CHEMBL1794249,5.01,=,=,EC50,nM,9870.0,CHEMBL1741171,EC50,uM,9.87,9870.0,5.005683
90,6720438,CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,COc1cc(C(c2c(C)n[nH]c2O)c2c(C)n[nH]c2O)cc(Br)c1OC,CHEMBL1201862,CHEMBL1735239,6.56,=,=,EC50,nM,274.0,CHEMBL1741171,EC50,uM,0.274,274.0,6.562249
91,6720439,CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,COc1cc(C2NC(=O)C(C#N)=C(Nc3cccc(C)c3)S2)ccc1O,CHEMBL1201862,CHEMBL1323586,4.70,=,=,EC50,nM,20000.0,CHEMBL1741171,EC50,uM,20.0,20000.0,4.698970
92,6720440,CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,O=C(NCc1ccccc1)c1nc(S(=O)(=O)Cc2ccccc2)ncc1Cl,CHEMBL1201862,CHEMBL1403320,5.31,=,=,EC50,nM,4880.0,CHEMBL1741171,EC50,uM,4.88,4880.0,5.311580


## 5. Retrieve molecule metadata

This optional but useful section retrieves molecule-level metadata and physicochemical properties from ChEMBL.


In [14]:
def chunked(values: list[str], size: int) -> Iterable[list[str]]:
    for index in range(0, len(values), size):
        yield values[index:index + size]


def fetch_molecule_metadata(molecule_ids: list[str]) -> pd.DataFrame:
    """Retrieve ChEMBL molecule metadata in batches."""
    if not molecule_ids:
        return pd.DataFrame()

    fields = [
        "molecule_chembl_id",
        "pref_name",
        "max_phase",
        "molecule_type",
        "structure_type",
        "molecule_structures",
        "molecule_properties",
    ]

    all_records = []
    molecule_client = new_client.molecule

    for batch in chunked(molecule_ids, MOLECULE_BATCH_SIZE):
        batch_records = list(
            molecule_client
            .filter(molecule_chembl_id__in=batch)
            .only(fields)
        )
        all_records.extend(batch_records)

    rows = []
    for record in all_records:
        structures = record.get("molecule_structures") or {}
        properties = record.get("molecule_properties") or {}

        rows.append({
            "molecule_chembl_id": record.get("molecule_chembl_id"),
            "molecule_pref_name": record.get("pref_name"),
            "max_phase": record.get("max_phase"),
            "molecule_type": record.get("molecule_type"),
            "structure_type": record.get("structure_type"),
            "metadata_canonical_smiles": structures.get("canonical_smiles"),
            "standard_inchi_key": structures.get("standard_inchi_key"),
            "full_mwt": properties.get("full_mwt"),
            "alogp": properties.get("alogp"),
            "psa": properties.get("psa"),
            "hba": properties.get("hba"),
            "hbd": properties.get("hbd"),
            "rtb": properties.get("rtb"),
            "aromatic_rings": properties.get("aromatic_rings"),
            "cx_logp": properties.get("cx_logp"),
        })

    metadata = (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["molecule_chembl_id"])
        .reset_index(drop=True)
    )

    metadata.to_csv(OUTPUT_DIR / "recA_molecule_metadata.csv", index=False)
    return metadata


molecule_ids = (
    ec50_activities["molecule_chembl_id"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

molecule_metadata = fetch_molecule_metadata(molecule_ids)

print(f"Unique molecules with metadata: {len(molecule_metadata)}")
molecule_metadata.head()


Unique molecules with metadata: 1909


,molecule_chembl_id,molecule_pref_name,max_phase,molecule_type,structure_type,metadata_canonical_smiles,standard_inchi_key,full_mwt,alogp,psa,hba,hbd,rtb,aromatic_rings,cx_logp
0,CHEMBL88326,None,None,Small molecule,MOL,Nc1ccc(O)c2c1C(=O)c1ccccc1C2=O,AQXYVFBSOOBBQV-UHFFFAOYSA-N,239.23,1.75,80.39,4.0,2.0,0.0,2.0,None
1,CHEMBL390559,None,None,Small molecule,MOL,Cc1nn(-c2ccccc2)c(Cl)c1/C=C1\SC(=S)N(CCCCCC(=O...,VIZMVONYWAFVSZ-VBKFSLOCSA-N,449.99,4.68,75.43,6.0,1.0,8.0,2.0,None
2,CHEMBL505618,None,None,Small molecule,MOL,CC(Nc1c[nH]c(=O)[nH]c1=O)c1ccccc1,PVIKSBZFUSEAMF-UHFFFAOYSA-N,231.25,1.24,77.75,3.0,3.0,3.0,2.0,None
3,CHEMBL604321,None,None,Small molecule,MOL,CCCNC1=C(N(C(C)=O)c2cccc(F)c2)C(=O)c2ccccc2C1=O,SAPBQULMOGDOGL-UHFFFAOYSA-N,366.39,3.47,66.48,4.0,1.0,5.0,2.0,None
4,CHEMBL586029,None,None,Small molecule,MOL,CCOC(=O)C1CCN(C(=O)c2c(C)oc3c2C(=O)c2ccccc2C3=...,YCQRDSJRASINSW-UHFFFAOYSA-N,395.41,2.78,93.89,6.0,0.0,3.0,2.0,None


## 6. Data processing and bioactivity labeling

This section keeps the original structure of the dengue NS3 notebook but applies it correctly to RecA EC50 data.

Labeling rule used here:

- `active`: EC50 ≤ 1,000 nM
- `intermediate`: 1,000 nM < EC50 < 10,000 nM
- `inactive`: EC50 ≥ 10,000 nM

For machine-learning classification, intermediate compounds are removed to form a clearer binary dataset.


In [15]:
def process_and_label_ec50(ec50_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Clean, deduplicate, label, and generate binary RecA EC50 datasets."""
    required_columns = ["molecule_chembl_id", "canonical_smiles", "standard_value"]
    missing = [column for column in required_columns if column not in ec50_df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = ec50_df.copy()
    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

    nonnull = df[
        df["standard_value"].notna()
        & df["canonical_smiles"].notna()
    ].copy()

    nonnull["normalized_smiles"] = nonnull["canonical_smiles"].astype(str).str.strip()

    deduplicated = (
        nonnull
        .sort_values(["standard_value", "molecule_chembl_id"], ascending=[True, True])
        .drop_duplicates(subset=["normalized_smiles"], keep="first")
        .reset_index(drop=True)
    )

    curated = deduplicated[
        ["molecule_chembl_id", "canonical_smiles", "standard_value", "standard_value_nM", "pEC50"]
    ].copy()

    curated["bioactivity_class"] = "intermediate"
    curated.loc[curated["standard_value"] <= ACTIVE_THRESHOLD_NM, "bioactivity_class"] = "active"
    curated.loc[curated["standard_value"] >= INACTIVE_THRESHOLD_NM, "bioactivity_class"] = "inactive"

    binary = curated[curated["bioactivity_class"].isin(["active", "inactive"])].copy()
    binary["class"] = binary["bioactivity_class"].map({"inactive": 0, "active": 1}).astype(int)

    active = binary[binary["bioactivity_class"] == "active"].copy()
    inactive = binary[binary["bioactivity_class"] == "inactive"].copy()

    target_n = min(len(active), len(inactive))
    if target_n > 0:
        balanced = (
            pd.concat([
                active.sample(n=target_n, random_state=RANDOM_STATE),
                inactive.sample(n=target_n, random_state=RANDOM_STATE),
            ], axis=0)
            .sample(frac=1, random_state=RANDOM_STATE)
            .reset_index(drop=True)
        )
    else:
        balanced = pd.DataFrame(columns=binary.columns)

    return nonnull, deduplicated, curated, binary, balanced


nonnull, deduplicated, curated, binary, balanced = process_and_label_ec50(ec50_activities)

nonnull.to_csv(OUTPUT_DIR / "recA_ec50_nonnull.csv", index=False)
deduplicated.to_csv(OUTPUT_DIR / "recA_ec50_deduplicated.csv", index=False)
curated.to_csv(OUTPUT_DIR / "recA_ec50_labeled.csv", index=False)
binary.to_csv(OUTPUT_DIR / "recA_ec50_binary.csv", index=False)
balanced.to_csv(OUTPUT_DIR / "recA_ec50_binary_balanced_50_50.csv", index=False)

print(f"After removing missing values : {len(nonnull)}")
print(f"After SMILES deduplication    : {len(deduplicated)}")
print(f"Final binary dataset          : {len(binary)}")
print(f"Balanced 50:50 dataset        : {len(balanced)}")

print("\nBioactivity class counts:")
print(curated["bioactivity_class"].value_counts().to_string())

curated.head()


After removing missing values : 1912
After SMILES deduplication    : 1907
Final binary dataset          : 915
Balanced 50:50 dataset        : 628

Bioactivity class counts:
bioactivity_class
intermediate    992
inactive        601
active          314


,molecule_chembl_id,canonical_smiles,standard_value,standard_value_nM,pEC50,bioactivity_class
0,CHEMBL1463659,Cn1nc(-c2ccccn2)nc2c(=O)n(C)c(=O)nc1-2,19.1,19.1,7.718967,active
1,CHEMBL3195749,COC(=O)c1ccccc1NC1=C/C(=N\S(=O)(=O)c2cccs2)c2c...,20.8,20.8,7.681937,active
2,CHEMBL601757,Cc1nc2c(=O)n(C)c(=O)nc-2n(C)n1,20.9,20.9,7.679854,active
3,CHEMBL1334062,Cn1nc(-c2ccc(Cl)cc2)nc2c(=O)n(C)c(=O)nc1-2,30.5,30.5,7.515700,active
4,CHEMBL1370184,CC(=O)Oc1c(/C=C(\C#N)C(N)=O)[nH]c2ccccc12,31.4,31.4,7.503070,active


## 7. Create publication-style data summary and final ML dataset

In [17]:
def create_activity_summary(curated_df: pd.DataFrame) -> pd.DataFrame:
    """Create one-row-per-compound activity summary for modeling/reporting."""
    summary = (
        curated_df
        .groupby("molecule_chembl_id", as_index=False)
        .agg(
            canonical_smiles=("canonical_smiles", "first"),
            median_ec50_nM=("standard_value", "median"),
            min_ec50_nM=("standard_value", "min"),
            max_ec50_nM=("standard_value", "max"),
            median_pEC50=("pEC50", "median"),
            bioactivity_class=("bioactivity_class", "first"),
        )
        .sort_values(["median_pEC50", "min_ec50_nM"], ascending=[False, True])
        .reset_index(drop=True)
    )

    return summary


activity_summary = create_activity_summary(curated)
activity_summary.to_csv(OUTPUT_DIR / "recA_activity_summary_ml.csv", index=False)

ml_dataset = activity_summary.merge(
    molecule_metadata,
    on="molecule_chembl_id",
    how="left",
)

ml_dataset.to_csv(OUTPUT_DIR / "recA_ml_dataset.csv", index=False)

data_curation_summary = pd.DataFrame([
    {"step": "Raw ChEMBL activity records", "records": len(raw_activities)},
    {"step": "Exact EC50 records in nM", "records": len(ec50_activities)},
    {"step": "After removing missing standard_value/canonical_smiles", "records": len(nonnull)},
    {"step": "After SMILES deduplication", "records": len(deduplicated)},
    {"step": "Active compounds", "records": int((curated["bioactivity_class"] == "active").sum())},
    {"step": "Intermediate compounds", "records": int((curated["bioactivity_class"] == "intermediate").sum())},
    {"step": "Inactive compounds", "records": int((curated["bioactivity_class"] == "inactive").sum())},
    {"step": "Final binary active/inactive dataset", "records": len(binary)},
    {"step": "Balanced 50:50 modeling dataset", "records": len(balanced)},
])

data_curation_summary.to_csv(OUTPUT_DIR / "01_data_curation_summary.csv", index=False)

data_curation_summary


,step,records
0,Raw ChEMBL activity records,2211
1,Exact EC50 records in nM,1914
2,After removing missing standard_value/canonica...,1912
3,After SMILES deduplication,1907
4,Active compounds,314
5,Intermediate compounds,992
6,Inactive compounds,601
7,Final binary active/inactive dataset,915
8,Balanced 50:50 modeling dataset,628


## 8. Download outputs in Google Colab

In [18]:
# Optional: run this cell in Google Colab to zip and download all output files.
try:
    from google.colab import files
    import shutil

    zip_path = shutil.make_archive("recA_chembl_outputs", "zip", OUTPUT_DIR)
    files.download(zip_path)
except Exception as error:
    print("Download is only available in Google Colab.")
    print(f"Output files are saved in: {OUTPUT_DIR.resolve()}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Expected output files

After running all cells, the main files are:

- `recA_target_search_hits.csv`
- `recA_selected_target.csv`
- `recA_activities_raw.csv`
- `recA_activities_ec50_exact.csv`
- `recA_molecule_metadata.csv`
- `recA_ec50_nonnull.csv`
- `recA_ec50_deduplicated.csv`
- `recA_ec50_labeled.csv`
- `recA_ec50_binary.csv`
- `recA_ec50_binary_balanced_50_50.csv`
- `recA_activity_summary_ml.csv`
- `recA_ml_dataset.csv`
- `01_data_curation_summary.csv`

This version is cleaner than the previous combined file because it avoids repeated `main()` functions, keeps the RecA target explicit, and preserves the step-by-step style of the original NS3 notebook.
